# 03a — Disability (TS038) EDA: Factor 9 Integration

**Purpose:** Load Census 2021 TS038 (disability), derive metrics, validate against master LSOA table,
compute correlations, and integrate disability as Factor 9 into `data/audit/master_lsoa_table.parquet`.

**Source:** `data/raw/census/census2021-ts038-lsoa.csv` (35,672 rows, England + Wales)
**Output:** Updated `data/audit/master_lsoa_table.parquet` with `disability_pct` and `disability_severe_pct`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from loguru import logger

PROJECT_ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
TS038_PATH = PROJECT_ROOT / "data/raw/census/census2021-ts038-lsoa.csv"
MASTER_PATH = PROJECT_ROOT / "data/audit/master_lsoa_table.parquet"
FIG_PATH = PROJECT_ROOT / "data/audit/fig_03a_disability_distribution.png"

checks: list[tuple[str, bool, str]] = []

## 1. Load and Profile TS038

In [2]:
df = pd.read_csv(TS038_PATH)
logger.info(f"Loaded TS038: {df.shape[0]} rows × {df.shape[1]} cols")

print("=== Column inventory ===")
for col in df.columns:
    n_null = df[col].isna().sum()
    dtype = df[col].dtype
    if dtype == object:
        n_unique = df[col].nunique()
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, unique={n_unique}")
    else:
        print(f"  {col!r}: dtype={dtype}, nulls={n_null}, min={df[col].min()}, max={df[col].max()}")

2026-03-13 00:01:25.838 | INFO     | __main__:<module>:2 - Loaded TS038: 35672 rows × 10 cols


=== Column inventory ===
  'date': dtype=int64, nulls=0, min=2021, max=2021
  'geography': dtype=str, nulls=0, min=Adur 001A, max=York 024F
  'geography code': dtype=str, nulls=0, min=E01000001, max=W01002040
  'Disability: Total: All usual residents': dtype=int64, nulls=0, min=999, max=9899
  'Disability: Disabled under the Equality Act': dtype=int64, nulls=0, min=46, max=1419
  'Disability: Disabled under the Equality Act: Day-to-day activities limited a lot': dtype=int64, nulls=0, min=5, max=574
  'Disability: Disabled under the Equality Act: Day-to-day activities limited a little': dtype=int64, nulls=0, min=33, max=1262
  'Disability: Not disabled under the Equality Act': dtype=int64, nulls=0, min=657, max=8480
  'Disability: Not disabled under the Equality Act: Has long term physical or mental health condition but day-to-day activities are not limited': dtype=int64, nulls=0, min=23, max=938
  'Disability: Not disabled under the Equality Act: No long term physical or mental health 

In [3]:
print("\n=== Head ===")
print(df.head(3).to_string())


=== Head ===
   date            geography geography code  Disability: Total: All usual residents  Disability: Disabled under the Equality Act  Disability: Disabled under the Equality Act: Day-to-day activities limited a lot  Disability: Disabled under the Equality Act: Day-to-day activities limited a little  Disability: Not disabled under the Equality Act  Disability: Not disabled under the Equality Act: Has long term physical or mental health condition but day-to-day activities are not limited  Disability: Not disabled under the Equality Act: No long term physical or mental health conditions
0  2021  City of London 001A      E01000001                                    1475                                          152                                                                                36                                                                                  116                                             1323                                                       

## 2. Null Check

In [4]:
total_nulls = df.isna().sum().sum()
null_ok = total_nulls == 0
checks.append(("Zero nulls in raw TS038", null_ok, f"Total nulls: {total_nulls}"))
assert null_ok, f"Expected 0 nulls, got {total_nulls}"
logger.info(f"Null check: {total_nulls} nulls — PASS")

2026-03-13 00:01:25.853 | INFO     | __main__:<module>:5 - Null check: 0 nulls — PASS


## 3. Filter to England

In [5]:
df_eng = df[df["geography code"].str.startswith("E")].copy().reset_index(drop=True)
n_eng = len(df_eng)
logger.info(f"England-only rows: {n_eng}")

eng_count_ok = n_eng == 33_755
checks.append(("England LSOA count = 33,755", eng_count_ok, f"Got {n_eng}"))
assert eng_count_ok, f"Expected 33,755 England LSOAs, got {n_eng}"

2026-03-13 00:01:25.861 | INFO     | __main__:<module>:3 - England-only rows: 33755


## 4. Derive Metrics

In [6]:
# Shorten column names for convenience
COL_TOTAL = "Disability: Total: All usual residents"
COL_DISABLED = "Disability: Disabled under the Equality Act"
COL_LIMITED_LOT = "Disability: Disabled under the Equality Act: Day-to-day activities limited a lot"

df_eng["disability_pct"] = (
    df_eng[COL_DISABLED] / df_eng[COL_TOTAL] * 100
).round(2)

df_eng["disability_severe_pct"] = (
    df_eng[COL_LIMITED_LOT] / df_eng[COL_TOTAL] * 100
).round(2)

print("\n=== Derived metrics — descriptive stats ===")
print(df_eng[["disability_pct", "disability_severe_pct"]].describe().round(2))

# Check no NaN in derived metrics
derived_nulls = df_eng[["disability_pct", "disability_severe_pct"]].isna().sum().sum()
derived_ok = derived_nulls == 0
checks.append(("Zero nulls in derived disability metrics", derived_ok, f"Nulls: {derived_nulls}"))
assert derived_ok


=== Derived metrics — descriptive stats ===
       disability_pct  disability_severe_pct
count        33755.00               33755.00
mean            17.49                   7.42
std              4.85                   3.04
min              1.81                   0.20
25%             13.96                   5.21
50%             17.00                   6.91
75%             20.57                   9.15
max             44.68                  26.06


## 5. Distribution Analysis

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for i, (col, label) in enumerate([
    ("disability_pct", "Disability % (any)"),
    ("disability_severe_pct", "Disability % (limited a lot)"),
]):
    # Histogram
    axes[i, 0].hist(df_eng[col], bins=60, color="#2563EB", edgecolor="none", alpha=0.85)
    axes[i, 0].set_title(f"{label} — Distribution")
    axes[i, 0].set_xlabel("Percentage (%)")
    axes[i, 0].set_ylabel("LSOA count")
    axes[i, 0].axvline(df_eng[col].mean(), color="red", linestyle="--", label=f"Mean: {df_eng[col].mean():.1f}%")
    axes[i, 0].legend()

    # Boxplot
    axes[i, 1].boxplot(df_eng[col], vert=True, patch_artist=True,
                        boxprops=dict(facecolor="#BFDBFE"))
    axes[i, 1].set_title(f"{label} — Boxplot")
    axes[i, 1].set_ylabel("Percentage (%)")
    axes[i, 1].set_xticks([])

plt.suptitle("TS038 Disability Metrics — England LSOAs (n=33,755)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"Figure saved: {FIG_PATH}")

/var/folders/xn/k_xsdymn32j0zkbwkzzdjlf80000gn/T/ipykernel_39774/14883404.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-13 00:01:26.184 | INFO     | __main__:<module>:26 - Figure saved: /Users/souravamseekarmarti/Projects/aequitas/data/audit/fig_03a_disability_distribution.png


## 6. Cross-validate Against Master LSOA Table

In [8]:
master = pd.read_parquet(MASTER_PATH)
logger.info(f"Master table: {master.shape[0]} rows × {master.shape[1]} cols")

master_rows_ok = len(master) == 33_755
checks.append(("Master table has 33,755 rows", master_rows_ok, f"Got {len(master)}"))

# Check population alignment — compare TS038 total vs master population for overlapping LSOAs
df_eng_coded = df_eng.rename(columns={"geography code": "lsoa_cd"})
overlap = df_eng_coded[["lsoa_cd", COL_TOTAL]].merge(
    master[["lsoa_cd", "population"]], on="lsoa_cd", how="inner"
)
logger.info(f"Overlap count: {len(overlap)}")

pop_align_ok = len(overlap) == 33_755
checks.append(("100% LSOA code overlap between TS038 and master table", pop_align_ok, f"Overlap: {len(overlap)}"))

corr_pop = overlap[COL_TOTAL].corr(overlap["population"])
logger.info(f"Population correlation (TS038 total vs master population): {corr_pop:.4f}")
pop_corr_ok = corr_pop > 0.95
checks.append((f"Population correlation TS038 vs master > 0.95", pop_corr_ok, f"r={corr_pop:.4f}"))

print(f"\nPopulation Pearson correlation: {corr_pop:.4f}")

2026-03-13 00:01:26.239 | INFO     | __main__:<module>:2 - Master table: 33755 rows × 48 cols


2026-03-13 00:01:26.245 | INFO     | __main__:<module>:12 - Overlap count: 33755


2026-03-13 00:01:26.246 | INFO     | __main__:<module>:18 - Population correlation (TS038 total vs master population): 1.0000



Population Pearson correlation: 1.0000


## 7. Correlation — Disability vs Other Factors

In [9]:
# Build disability lookup
disability_lookup = df_eng_coded[["lsoa_cd", "disability_pct", "disability_severe_pct"]]

master_w_dis = master.merge(disability_lookup, on="lsoa_cd", how="left")

numeric_factors = [
    "imd_score", "income_score", "employment_score", "health_score",
    "unemployment_rate", "elderly_pct", "nocar_pct", "stops_per_1k",
    "nonwhite_pct", "bus_need_score",
]

print("\n=== Pearson correlations with disability_pct ===")
corr_results = {}
for col in numeric_factors:
    if col in master_w_dis.columns:
        r = master_w_dis["disability_pct"].corr(master_w_dis[col])
        corr_results[col] = round(r, 4)
        print(f"  {col:<25}: r = {r:+.4f}")

print("\n=== Pearson correlations with disability_severe_pct ===")
for col in numeric_factors:
    if col in master_w_dis.columns:
        r = master_w_dis["disability_severe_pct"].corr(master_w_dis[col])
        print(f"  {col:<25}: r = {r:+.4f}")


=== Pearson correlations with disability_pct ===
  imd_score                : r = +0.5441
  income_score             : r = +0.4414
  employment_score         : r = +0.6505
  health_score             : r = +0.6859
  unemployment_rate        : r = +0.2669
  elderly_pct              : r = +0.3316
  nocar_pct                : r = +0.1811
  stops_per_1k             : r = +0.1438
  nonwhite_pct             : r = -0.3666
  bus_need_score           : r = +0.4752

=== Pearson correlations with disability_severe_pct ===
  imd_score                : r = +0.6583
  income_score             : r = +0.5811
  employment_score         : r = +0.7530
  health_score             : r = +0.7541
  unemployment_rate        : r = +0.3956
  elderly_pct              : r = +0.1954
  nocar_pct                : r = +0.2968
  stops_per_1k             : r = +0.0758
  nonwhite_pct             : r = -0.1746
  bus_need_score           : r = +0.5355


## 8. Update Master Table

In [10]:
# Merge disability into master
master_updated = master.merge(
    disability_lookup[["lsoa_cd", "disability_pct", "disability_severe_pct"]],
    on="lsoa_cd",
    how="left",
)

# Assert 100% join — zero nulls after merge
dis_nulls_after = master_updated[["disability_pct", "disability_severe_pct"]].isna().sum().sum()
join_ok = dis_nulls_after == 0
checks.append(("Zero nulls after merging disability into master table", join_ok, f"Nulls: {dis_nulls_after}"))
assert join_ok, f"Expected 0 nulls after merge, got {dis_nulls_after}"

row_count_ok = len(master_updated) == 33_755
checks.append(("Master table row count preserved = 33,755", row_count_ok, f"Got {len(master_updated)}"))
assert row_count_ok

master_updated.to_parquet(MASTER_PATH, index=False)
logger.info(f"Master table updated: {len(master_updated)} rows × {len(master_updated.columns)} cols → {MASTER_PATH}")
print(f"\nMaster table columns ({len(master_updated.columns)}):")
print(master_updated.columns.tolist())

2026-03-13 00:01:26.313 | INFO     | __main__:<module>:19 - Master table updated: 33755 rows × 50 cols → /Users/souravamseekarmarti/Projects/aequitas/data/audit/master_lsoa_table.parquet



Master table columns (50):
['lsoa_cd', 'lsoa_nm', 'lad_cd', 'lad_nm', 'imd_score', 'imd_rank', 'imd_decile', 'income_score', 'employment_score', 'education_score', 'health_score', 'crime_score', 'barriers_score', 'living_env_score', 'idaci_score', 'idaopi_score', 'pop_imd_2022', 'population', 'pop_16plus', 'econ_active', 'unemployed', 'retired', 'longterm_sick', 'unemployment_rate', 'elderly_65plus', 'elderly_pct', 'young_0_14', 'young_pct', 'working_age_pct', 'total_hh', 'nocar_hh', 'nocar_pct', 'nonwhite_pct', 'eth_white', 'eth_asian', 'eth_black', 'eth_mixed', 'eth_other', 'eth_total', 'ruc_code', 'ruc_name', 'urban_rural', 'stop_count', 'region', 'car_need_category', 'stops_per_1k', 'has_bus', 'bus_need_score', 'disability_pct', 'disability_severe_pct']


## 9. Validation Summary

In [11]:
print("\n" + "=" * 60)
print("VALIDATION SUMMARY — 03a_disability_ts038")
print("=" * 60)

fail_count = 0
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        fail_count += 1
    print(f"  [{status}] {name}")
    if detail:
        print(f"         → {detail}")

print("-" * 60)
print(f"  Total checks: {len(checks)}")
print(f"  FAILs:        {fail_count}")
print("=" * 60)

# Key figures for figures-registry.md
mean_dis = master_updated["disability_pct"].mean()
min_dis = master_updated["disability_pct"].min()
max_dis = master_updated["disability_pct"].max()
print(f"\nKey figures:")
print(f"  ST-008  disability_pct mean:  {mean_dis:.2f}%")
print(f"  ST-009  disability_pct range: {min_dis:.2f}%–{max_dis:.2f}%")

assert fail_count == 0, f"{fail_count} FAILs — see summary above"
logger.success("All checks passed. Factor 9 (disability) integrated into master LSOA table.")

2026-03-13 00:01:26.318 | SUCCESS  | __main__:<module>:28 - All checks passed. Factor 9 (disability) integrated into master LSOA table.



VALIDATION SUMMARY — 03a_disability_ts038
  [PASS] Zero nulls in raw TS038
         → Total nulls: 0
  [PASS] England LSOA count = 33,755
         → Got 33755
  [PASS] Zero nulls in derived disability metrics
         → Nulls: 0
  [PASS] Master table has 33,755 rows
         → Got 33755
  [PASS] 100% LSOA code overlap between TS038 and master table
         → Overlap: 33755
  [PASS] Population correlation TS038 vs master > 0.95
         → r=1.0000
  [PASS] Zero nulls after merging disability into master table
         → Nulls: 0
  [PASS] Master table row count preserved = 33,755
         → Got 33755
------------------------------------------------------------
  Total checks: 8
  FAILs:        0

Key figures:
  ST-008  disability_pct mean:  17.49%
  ST-009  disability_pct range: 1.81%–44.68%
